# 02_preprocessing

**Objective:** Preprocessing and transformations





# 02 — Preprocessing

## 02.1 — Feature Selection and Prediction-Time Eligibility

**Objective:** Predict loan charge-off risk using information
available at the completed application and credit-bureau
assessment stage, before lender pricing and funding.

- Target: Fully Paid = 0; Charged Off = 1.
- Source: 2013–2016 resolved, 36-month raw loan cohort.
- Retain 50 numerical EDA candidates provisionally.
- Review six categorical candidates for eligibility.
- Exclude post-loan data, IDs, lender-generated outcomes,
  late-rollout fields and previously documented exclusions.
- Preserve 2016 as an untouched chronological holdout.

No preprocessing transformations or feature selection
are fitted at this stage.


In [1]:

# 02.1 — Load EDA feature manifest and validate candidates

from pathlib import Path
import pandas as pd
import polars as pl
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SOURCE = ROOT / "data/interim/lendingclub_2013_2016_36m_raw.parquet"
MANIFEST_PATH = ROOT / "reports/eda/01_feature_manifest_draft.csv"

assert SOURCE.exists(), f"Dataset not found: {SOURCE}"
assert MANIFEST_PATH.exists(), (
    f"Feature manifest not found: {MANIFEST_PATH}\n"
    "Run the final feature-manifest export cell in Notebook 01."
)

# 'decisions' is the Notebook 02 DataFrame
# 'manifest' belongs to Notebook 01
decisions = pd.read_csv(MANIFEST_PATH)

required_columns = {"feature", "data_group", "model_status"}
assert required_columns.issubset(decisions.columns)
assert decisions["feature"].is_unique

# Numerical candidates
numeric_features = decisions.loc[
    (decisions["data_group"] == "NUMERICAL")
    & (decisions["model_status"] == "CANDIDATE_NUMERIC"),
    "feature"
].tolist()

# All six categorical candidates, including pending eligibility reviews
categorical_statuses = [
    "CANDIDATE_CATEGORICAL",
    "REVIEW_GEOGRAPHY",
    "REVIEW_HISTORICAL_AVAILABILITY",
    "HOLD_VERIFICATION",
]

categorical_features = decisions.loc[
    (decisions["data_group"] == "CATEGORICAL")
    & decisions["model_status"].isin(categorical_statuses),
    "feature"
].tolist()

# Verify candidate counts from EDA
assert len(numeric_features) == 50
assert len(categorical_features) == 6

# Validate against actual dataset columns without loading all rows
schema = pl.scan_parquet(SOURCE).collect_schema()
required_features = set(numeric_features + categorical_features)

missing = required_features - set(schema.names())
assert not missing, f"Missing dataset features: {sorted(missing)}"
assert {"target", "issue_date"} <= set(schema.names())

print("Numerical candidates:", len(numeric_features))
print("Categorical candidates:", len(categorical_features))
print("Total candidates:", len(required_features))
print("Dataset schema: PASS")

print("\nCategorical prediction-time review:")
display(
    decisions.loc[
        decisions["feature"].isin(categorical_features),
        ["feature", "model_status"]
    ].sort_values("feature").reset_index(drop=True)
)


Numerical candidates: 50
Categorical candidates: 6
Total candidates: 56
Dataset schema: PASS

Categorical prediction-time review:


,feature,model_status
0,addr_state,REVIEW_GEOGRAPHY
1,application_type,REVIEW_HISTORICAL_AVAILABILITY
2,emp_length,CANDIDATE_CATEGORICAL
3,home_ownership,CANDIDATE_CATEGORICAL
4,purpose,CANDIDATE_CATEGORICAL
5,verification_status,HOLD_VERIFICATION


In [2]:

# 02.2 — Update prediction-time feature decisions

feature_updates = {
    "home_ownership": "APPROVED_CATEGORICAL",
    "addr_state": "REVIEW_BASELINE_CONTRIBUTION",
    "emp_length": "ENGINEER_NUMERIC",
    "purpose": "EXCLUDE_PRIMARY",
    "verification_status": "EXCLUDE_PRIMARY",
    "application_type": "EXCLUDE_PRIMARY",
}

for feature, status in feature_updates.items():
    assert feature in decisions["feature"].values
    decisions.loc[
        decisions["feature"] == feature, "model_status"
    ] = status

# Keep the existing 50 numerical candidates
ordinal_features = ["emp_length"]

# Categorical inputs for the initial baseline
categorical_features = ["home_ownership", "addr_state"]

assert len(numeric_features) == 50
assert len(ordinal_features) == 1
assert len(categorical_features) == 2

# Preserve decisions for the future .py pipeline
REPORT_DIR = ROOT / "reports/preprocessing"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DECISIONS_PATH = REPORT_DIR / "02_feature_manifest_draft.csv"
decisions.to_csv(DECISIONS_PATH, index=False)

print("Original numerical:", len(numeric_features))
print("Ordinal conversion:", ordinal_features)
print("Categorical:", categorical_features)
print("Total inputs:", 53)
print("Saved:", DECISIONS_PATH)


Original numerical: 50
Ordinal conversion: ['emp_length']
Categorical: ['home_ownership', 'addr_state']
Total inputs: 53
Saved: d:\ML_Projects\P2P-Lending-Default-Prediction\reports\preprocessing\02_feature_manifest_draft.csv



## 02.3 — Pre-Split Target Distribution

Examine loan counts, default counts and default rates:

1. By issue year (2013–2016).
2. By proposed train, validation and test periods.

Verify that both classes are represented and assess
temporal changes in default prevalence.

The 2016 test outcomes are examined only for descriptive
validation, not model selection or tuning.


In [3]:

# 02.3 — Target distribution before finalizing splits

from datetime import date

lf = pl.scan_parquet(SOURCE)

def target_summary(data, group_col):
    return (
        data.group_by(group_col)
        .agg(
            pl.len().alias("total"),
            (pl.col("target") == 0).sum().alias("fully_paid"),
            (pl.col("target") == 1).sum().alias("charged_off"),
        )
        .with_columns(
            (100 * pl.col("charged_off") / pl.col("total"))
            .round(2).alias("default_pct")
        )
        .sort(group_col)
        .collect()
        .to_pandas()
    )


# A. Target distribution by year
yearly = target_summary(lf, "issue_year")

print("A. YEAR-WISE TARGET DISTRIBUTION")
display(yearly)


# B. Proposed chronological periods (audit only)
audit_lf = lf.with_columns(
    pl.when(pl.col("issue_date") < date(2015, 10, 1))
    .then(pl.lit("train"))
    .when(pl.col("issue_date") < date(2016, 1, 1))
    .then(pl.lit("validation"))
    .otherwise(pl.lit("test"))
    .alias("proposed_split")
)

split_distribution = target_summary(
    audit_lf, "proposed_split"
)

# Display chronologically
order = ["train", "validation", "test"]
split_distribution["proposed_split"] = pd.Categorical(
    split_distribution["proposed_split"],
    categories=order,
    ordered=True
)

split_distribution = split_distribution.sort_values(
    "proposed_split"
).reset_index(drop=True)

print("\nB. PROPOSED SPLIT TARGET DISTRIBUTION")
display(split_distribution)

# Basic consistency checks
assert yearly["total"].sum() == 869_540
assert (yearly["fully_paid"] + yearly["charged_off"]
        == yearly["total"]).all()

assert split_distribution["total"].sum() == 869_540
assert (split_distribution[["fully_paid", "charged_off"]] > 0).all().all()

print("\nTarget distribution checks: PASS")


A. YEAR-WISE TARGET DISTRIBUTION


,issue_year,total,fully_paid,charged_off,default_pct
0,2013,100422,88044,12378,12.33
1,2014,162570,140255,22315,13.73
2,2015,283173,240995,42178,14.89
3,2016,323375,272076,51299,15.86



B. PROPOSED SPLIT TARGET DISTRIBUTION


,proposed_split,total,fully_paid,charged_off,default_pct
0,train,457355,393667,63688,13.93
1,validation,88810,75627,13183,14.84
2,test,323375,272076,51299,15.86



Target distribution checks: PASS



## 02.4 — Freeze Chronological Train/Validation/Test Split

Based on the target distribution review, finalize:

- **Train:** 2013-01-01 to 2015-09-30
- **Validation:** 2015-10-01 to 2015-12-31
- **Test:** 2016-01-01 to 2016-12-31

The split is deterministic and based on loan issue date.

- Fit preprocessing transformations on training data only.
- Use validation data for model selection and tuning.
- Reserve test data for final evaluation.
- Do not randomly shuffle or stratify across time periods.

Retain the original Parquet unchanged. Save split metadata
for reproducibility and future Python pipeline implementation.


In [4]:

# 02.4 — Freeze chronological splits

from datetime import date

TRAIN_END = date(2015, 10, 1)
VAL_END = date(2016, 1, 1)
TEST_END = date(2017, 1, 1)

# Create deterministic split labels
split_lf = pl.scan_parquet(SOURCE).with_columns(
    pl.when(
        (pl.col("issue_date") >= date(2013, 1, 1))
        & (pl.col("issue_date") < TRAIN_END)
    ).then(pl.lit("train"))

    .when(
        (pl.col("issue_date") >= TRAIN_END)
        & (pl.col("issue_date") < VAL_END)
    ).then(pl.lit("validation"))

    .when(
        (pl.col("issue_date") >= VAL_END)
        & (pl.col("issue_date") < TEST_END)
    ).then(pl.lit("test"))

    .otherwise(pl.lit("outside_scope"))
    .alias("split")
)

# LazyFrames — no duplicated dataset files
train_lf = split_lf.filter(pl.col("split") == "train")
val_lf = split_lf.filter(pl.col("split") == "validation")
test_lf = split_lf.filter(pl.col("split") == "test")

# Validate split sizes and date boundaries
split_summary = (
    split_lf.group_by("split")
    .agg(
        pl.len().alias("rows"),
        pl.col("issue_date").min().alias("first_date"),
        pl.col("issue_date").max().alias("last_date"),
    )
    .collect()
    .to_pandas()
)

expected = {
    "train": 457355,
    "validation": 88810,
    "test": 323375,
}

actual = dict(zip(split_summary["split"], split_summary["rows"]))

assert actual == expected, f"Unexpected split sizes: {actual}"
assert sum(actual.values()) == 869540

# Sort for reporting
split_summary["split"] = pd.Categorical(
    split_summary["split"],
    categories=["train", "validation", "test"],
    ordered=True,
)

split_summary = split_summary.sort_values("split")

# Record the exact filtering rules
split_summary["start_inclusive"] = [
    "2013-01-01", "2015-10-01", "2016-01-01"
]
split_summary["end_exclusive"] = [
    "2015-10-01", "2016-01-01", "2017-01-01"
]

# Save compact metadata, not duplicate loan data
REPORT_DIR = ROOT / "reports/preprocessing"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

split_summary.to_csv(
    REPORT_DIR / "02_chronological_split_summary.csv",
    index=False
)

display(split_summary)

print("\nChronological split: VERIFIED")
print("Train, validation and test LazyFrames are ready.")


,split,rows,first_date,last_date,start_inclusive,end_exclusive
0,train,457355,2013-01-01,2015-09-01,2013-01-01,2015-10-01
2,validation,88810,2015-10-01,2015-12-01,2015-10-01,2016-01-01
1,test,323375,2016-01-01,2016-12-01,2016-01-01,2017-01-01



Chronological split: VERIFIED
Train, validation and test LazyFrames are ready.



## 02.5 — Special Feature Cleaning

Inspect and prepare three features using training data:

- `dti`: Convert to numeric; treat negative values as missing.
  Flag `999` for review rather than automatically removing it.
- `revol_util`: Remove `%` and convert to numeric percentage points.
  Preserve legitimate utilization above 100%.
- `emp_length`: Convert employment duration to numerical years.
  Map `< 1 year` to 0 and `10+ years` to 10.
  Preserve missing values.

These are deterministic transformations. No imputation,
scaling, outlier capping or row deletion is performed.


In [5]:

# 02.5 — Targeted type conversion and cleaning

# Parse raw string values
dti_raw = pl.col("dti").cast(pl.String).str.strip_chars()
dti_num = dti_raw.cast(pl.Float64, strict=False)

revol_raw = pl.col("revol_util").cast(pl.String).str.strip_chars()
revol_num = (
    revol_raw.str.replace_all("%", "")
    .cast(pl.Float64, strict=False)
)

emp_raw = pl.col("emp_length").cast(pl.String).str.strip_chars()

emp_years = (
    pl.when(emp_raw == "< 1 year").then(0)
    .when(emp_raw == "10+ years").then(10)
    .otherwise(
        emp_raw.str.extract(r"^(\d+)\s+years?$", 1)
        .cast(pl.Int64, strict=False)
    )
)

# Preview cleaned columns without replacing originals
train_parse_preview = train_lf.with_columns(
    pl.when(dti_num.is_finite() & (dti_num >= 0))
    .then(dti_num).otherwise(None).alias("_dti_clean"),

    pl.when(revol_num.is_finite())
    .then(revol_num).otherwise(None).alias("_revol_util_clean"),

    emp_years.alias("_emp_length_years")
)

# Summary of known anomalies and conversion results
summary = train_parse_preview.select(
    pl.len().alias("train_rows"),

    (dti_num < 0).fill_null(False).sum().alias("negative_dti"),
    (dti_num == 999).fill_null(False).sum().alias("dti_999"),

    pl.col("_dti_clean").null_count().alias("dti_missing"),
    pl.col("_revol_util_clean").null_count().alias("revol_missing"),

    (pl.col("_revol_util_clean") > 100)
    .fill_null(False).sum().alias("revol_above_100"),

    pl.col("_emp_length_years").null_count().alias("emp_missing")
).collect()

print("CLEANING SUMMARY — TRAIN ONLY")
display(summary.to_pandas())

# Verify employment-duration mapping
emp_summary = (
    train_parse_preview
    .group_by("emp_length")
    .agg(
        pl.len().alias("loans"),
        pl.col("_emp_length_years").first().alias("mapped_years")
    )
    .sort("loans", descending=True)
    .collect()
)

print("\nEMPLOYMENT LENGTH MAPPING")
display(emp_summary.to_pandas())


CLEANING SUMMARY — TRAIN ONLY


,train_rows,negative_dti,dti_999,dti_missing,revol_missing,revol_above_100,emp_missing
0,457355,0,0,0,221,1731,27145



EMPLOYMENT LENGTH MAPPING


,emp_length,loans,mapped_years
0,10+ years,144186,10.0
1,2 years,41407,2.0
2,3 years,36891,3.0
3,< 1 year,36557,0.0
4,1 year,30103,1.0
5,5 years,28095,5.0
6,,27145,NaN
7,4 years,26586,4.0
8,7 years,23790,7.0
9,8 years,22892,8.0



## 02.6 — Numerical Type Conversion Validation

Validate numeric parsing for all 50 selected numerical
features using training data only.

- Identify unexpected non-numeric values.
- Distinguish missing values from parsing failures.
- Record occurrences of -1 for later semantic review.
- Preserve raw data and genuine numerical extremes.

No imputation, scaling, capping or row deletion.


In [6]:

# 02.6 — Validate numeric parsing (training only)

numeric_audit_rows = []
missing_tokens = ["", "nan", "null", "none", "na", "n/a"]

for start in range(0, len(numeric_features), 10):
    batch = numeric_features[start:start + 10]
    expressions = []

    for c in batch:
        raw = pl.col(c).cast(pl.String).str.strip_chars()

        # Only revol_util requires percentage-symbol removal
        cleaned = (
            raw.str.replace_all("%", "")
            if c == "revol_util" else raw
        )

        parsed = cleaned.cast(pl.Float64, strict=False)

        missing = (
            raw.is_null()
            | raw.str.to_lowercase().is_in(missing_tokens)
        ).fill_null(True)

        invalid = (
            ~missing & ~parsed.is_finite().fill_null(False)
        )

        expressions.extend([
            missing.sum().alias(f"{c}__missing"),
            invalid.sum().alias(f"{c}__invalid"),
            (parsed == -1).fill_null(False)
            .sum().alias(f"{c}__minus_one"),
        ])

    result = train_lf.select(expressions).collect().row(
        0, named=True
    )

    for c in batch:
        numeric_audit_rows.append({
            "feature": c,
            "missing": result[f"{c}__missing"],
            "invalid_parse": result[f"{c}__invalid"],
            "minus_one": result[f"{c}__minus_one"],
        })

numeric_parse_audit = pd.DataFrame(numeric_audit_rows)

numeric_parse_audit["missing_pct"] = (
    100 * numeric_parse_audit["missing"] / 457355
).round(2)

numeric_parse_audit = numeric_parse_audit.sort_values(
    ["invalid_parse", "minus_one", "missing"],
    ascending=False
).reset_index(drop=True)

# Save a small reproducible quality report
REPORT_DIR.mkdir(parents=True, exist_ok=True)

numeric_parse_audit.to_csv(
    REPORT_DIR / "02_numeric_parse_audit.csv",
    index=False
)

print("Features checked:", len(numeric_parse_audit))
print("Features with parsing failures:",
      (numeric_parse_audit["invalid_parse"] > 0).sum())

print("\nFEATURES REQUIRING REVIEW")
display(
    numeric_parse_audit[
        (numeric_parse_audit["invalid_parse"] > 0)
        | (numeric_parse_audit["minus_one"] > 0)
    ]
)

print("\nTOP 10 FEATURES BY MISSINGNESS")
display(
    numeric_parse_audit.sort_values(
        "missing_pct", ascending=False
    ).head(10)
)


Features checked: 50
Features with parsing failures: 0

FEATURES REQUIRING REVIEW


,feature,missing,invalid_parse,minus_one,missing_pct



TOP 10 FEATURES BY MISSINGNESS


,feature,missing,invalid_parse,minus_one,missing_pct
0,mths_since_last_delinq,229705,0,0,50.22
1,mths_since_recent_inq,47831,0,0,10.46
2,mo_sin_old_il_acct,17249,0,0,3.77
3,num_tl_120dpd_2m,15741,0,0,3.44
4,bc_util,4784,0,0,1.05
5,percent_bc_gt_75,4728,0,0,1.03
6,bc_open_to_buy,4475,0,0,0.98
7,mths_since_recent_bc,4150,0,0,0.91
8,revol_util,221,0,0,0.05
9,pct_tl_nvr_dlq,114,0,0,0.02



## 02.7 — Numerical Cleaning and Missingness Indicators

Apply deterministic transformations consistently to train,
validation and test data.

- Convert the 50 numerical predictors from raw strings.
- Parse `revol_util` as percentage points.
- Treat negative DTI as missing.
- Convert `emp_length` to numerical years.
- Add missingness indicators for delinquency recency
  and employment length.

Preserve nulls until training-fitted imputation.
No rows are removed and no source files are modified.


In [18]:

# 02.7 — Reusable deterministic feature cleaning (corrected)
def prepare_features(data, include_engineered=False):
    expressions = []

    # Convert all 50 numerical features
    for c in numeric_features:
        raw = pl.col(c).cast(pl.String).str.strip_chars()

        if c == "revol_util":
            raw = raw.str.replace_all("%", "")

        value = raw.cast(pl.Float32, strict=False)

        if c == "dti":
            value = (
                pl.when(
                    value.is_finite().fill_null(False)
                    & (value >= 0)
                )
                .then(value)
                .otherwise(None)
            )

        expressions.append(value.alias(c))

    # Employment length -> numerical years
    emp = pl.col("emp_length").cast(pl.String).str.strip_chars()

    emp_years = (
        pl.when(emp == "< 1 year").then(0)
        .when(emp == "10+ years").then(10)
        .otherwise(
            emp.str.extract(r"^(\d+)\s+years?$", 1)
            .cast(pl.Int32, strict=False)
        )
    )

    expressions.append(emp_years.alias("emp_length_years"))

    # Keep selected categorical features
    for c in categorical_features:
        expressions.append(
            pl.col(c).cast(pl.String).str.strip_chars().alias(c)
        )

    # Target and date for evaluation
    expressions.extend(["target", "issue_date"])
    
    if include_engineered:
            credit_text = (
            pl.col("earliest_cr_line")
            .cast(pl.String)
            .str.strip_chars()
        )

            credit_date = (
            pl.concat_str([credit_text, pl.lit("-01")])
            .str.strptime(pl.Date, "%b-%Y-%d", strict=False)
        )

            age_months = (
            (pl.col("issue_date").dt.year() - credit_date.dt.year()) * 12
            + pl.col("issue_date").dt.month()
            - credit_date.dt.month()
        )

            expressions.append(
            pl.when(age_months >= 0)
            .then((age_months / 12).cast(pl.Float32))
            .otherwise(None)
            .alias("credit_history_years")
        )

    # First convert, THEN calculate missingness indicators
    prepared = data.select(expressions)

    prepared = prepared.with_columns(
        pl.col("mths_since_last_delinq")
        .is_null()
        .cast(pl.Int8)
        .alias("delinq_recency_missing"),

        pl.col("emp_length_years")
        .is_null()
        .cast(pl.Int8)
        .alias("emp_length_missing"),
        
        pl.col("mths_since_recent_inq")
        .is_null()
        .cast(pl.Int8)
        .alias("recent_inq_missing")
    )
    if include_engineered:
        prepared = prepared.with_columns(
            pl.when(
                (pl.col("annual_inc") > 0)
                & pl.col("annual_inc").is_finite()
                & pl.col("loan_amnt").is_finite()
            )
            .then(pl.col("loan_amnt") / pl.col("annual_inc"))
            .otherwise(None)
            .cast(pl.Float32)
            .alias("loan_to_income")
        )
    return prepared


# Apply consistently to all three splits
train_prepared_lf = prepare_features(train_lf)
val_prepared_lf = prepare_features(val_lf)
test_prepared_lf = prepare_features(test_lf)

# Verify training indicators
result = train_prepared_lf.select(
    pl.len().alias("rows"),
    pl.col("mths_since_last_delinq").null_count().alias("delinq_nulls"),
    pl.col("delinq_recency_missing").sum().alias("delinq_indicators"),
    pl.col("emp_length_years").null_count().alias("emp_nulls"),
    pl.col("emp_length_missing").sum().alias("emp_indicators"),
).collect()

display(result.to_pandas())

assert result["rows"][0] == 457355
assert result["delinq_nulls"][0] == result["delinq_indicators"][0]
assert result["emp_nulls"][0] == result["emp_indicators"][0]

print("PASS: Cleaning and missingness indicators verified.")
print("No imputation performed yet.")


,rows,delinq_nulls,delinq_indicators,emp_nulls,emp_indicators
0,457355,229705,229705,27145,27145


PASS: Cleaning and missingness indicators verified.
No imputation performed yet.



## 02.8 — Training-Fitted Numerical Imputation

Fit a median imputer using only the training period.

- Impute missing values in 50 numerical predictors and
  the engineered employment-length feature.
- Preserve delinquency and employment missingness indicators.
- Keep the same fitted statistics for validation and test.
- Do not modify the original Parquet files.

This establishes the initial imputation strategy for
the future preprocessing pipeline.


In [8]:

# 02.8 — Fit numerical imputer on training data only

import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_input_features = numeric_features + ["emp_length_years"]

indicator_features = [
    "delinq_recency_missing",
    "emp_length_missing",
]

# Collect only selected numerical training features
X_train_numeric = (
    train_prepared_lf
    .select(numeric_input_features)
    .collect()
    .to_pandas()
)

# Avoid silently dropping completely empty features
all_missing = X_train_numeric.columns[
    X_train_numeric.isna().all()
].tolist()

assert not all_missing, f"All-missing training features: {all_missing}"
assert len(numeric_input_features) == 51

# Define reusable sklearn pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Fit exclusively on training data
numeric_pipeline.fit(X_train_numeric)

# Record fitted statistics for reproducibility
imputation_report = pd.DataFrame({
    "feature": numeric_input_features,
    "train_missing": X_train_numeric.isna().sum().to_numpy(),
    "train_median": numeric_pipeline.named_steps[
        "imputer"
    ].statistics_,
})

imputation_report["missing_pct"] = (
    100 * imputation_report["train_missing"]
    / len(X_train_numeric)
).round(2)

imputation_report = imputation_report.sort_values(
    "missing_pct", ascending=False
).reset_index(drop=True)

# Save compact imputation report
REPORT_DIR.mkdir(parents=True, exist_ok=True)

imputation_report.to_csv(
    REPORT_DIR / "02_training_imputation_statistics.csv",
    index=False
)

print("Numerical features:", len(numeric_input_features))
print("Imputer fitted on training observations:", len(X_train_numeric))
print("Strategy:", numeric_pipeline.named_steps["imputer"].strategy)

print("\nTop missing features and fitted medians:")
display(imputation_report.head(12))

print("\nImputation fitted successfully — TRAIN ONLY")


Numerical features: 51
Imputer fitted on training observations: 457355
Strategy: median

Top missing features and fitted medians:


,feature,train_missing,train_median,missing_pct
0,mths_since_last_delinq,229705,31.000000,50.22
1,mths_since_recent_inq,47831,5.000000,10.46
2,emp_length_years,27145,6.000000,5.94
3,mo_sin_old_il_acct,17249,128.000000,3.77
4,num_tl_120dpd_2m,15741,0.000000,3.44
5,bc_util,4784,66.900002,1.05
6,percent_bc_gt_75,4728,50.000000,1.03
7,bc_open_to_buy,4475,3767.000000,0.98
8,mths_since_recent_bc,4150,14.000000,0.91
9,revol_util,221,54.799999,0.05



Imputation fitted successfully — TRAIN ONLY



## 02.9 — Inquiry-Recency Missingness Investigation

Investigate the 10.46% missingness in `mths_since_recent_inq`.

Compare missing recency against `inq_last_6mths`:

- Zero inquiries in the last six months.
- One or more inquiries in the last six months.
- Missing inquiry counts, if any.

Determine whether recency missingness appears event-related
or requires further investigation.

No imputation or feature removal is performed.


In [9]:

# 02.9 — Inquiry recency missingness vs recent inquiry counts

inquiry_review = (
    train_prepared_lf
    .with_columns(
        pl.when(pl.col("inq_last_6mths").is_null())
        .then(pl.lit("Count missing"))
        .when(pl.col("inq_last_6mths") == 0)
        .then(pl.lit("0 inquiries"))
        .when(pl.col("inq_last_6mths") > 0)
        .then(pl.lit("1+ inquiries"))
        .otherwise(pl.lit("Other"))
        .alias("inquiry_group")
    )
    .group_by("inquiry_group")
    .agg(
        pl.len().alias("loans"),
        pl.col("mths_since_recent_inq")
        .is_null().sum().alias("recency_missing"),
    )
    .with_columns(
        (
            100 * pl.col("recency_missing") / pl.col("loans")
        ).round(2).alias("missing_pct")
    )
    .sort("inquiry_group")
    .collect()
    .to_pandas()
)

display(inquiry_review)

print(
    "\nTotal recency missing:",
    inquiry_review["recency_missing"].sum()
)

assert inquiry_review["loans"].sum() == 457355
assert inquiry_review["recency_missing"].sum() == 47831


,inquiry_group,loans,recency_missing,missing_pct
0,0 inquiries,257156,47823,18.6
1,1+ inquiries,200199,8,0.0



Total recency missing: 47831



## 02.10 — Categorical Encoding

Prepare `home_ownership` and `addr_state`:

- Normalize blank and missing categories.
- Represent missing values with `MISSING`.
- Apply One-Hot Encoding without assuming an ordering.
- Handle unseen validation/test categories safely.
- Fit categorical preprocessing on training data only.

Retain `addr_state` provisionally for a validation-based
feature contribution comparison.

No category dropping or rare-category grouping yet.


In [10]:

# 02.10 — Training-fitted categorical preprocessing

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

# Stateless normalization, reusable in the future .py pipeline
def normalize_categories(X):
    X = X.copy()

    for c in X.columns:
        X[c] = X[c].astype("string").str.strip()
        X[c] = X[c].replace(
            {"": pd.NA, "nan": pd.NA, "null": pd.NA}
        )
        X[c] = X[c].astype(object).where(X[c].notna(), np.nan)

    return X


categorical_pipeline = Pipeline([
    ("normalize", FunctionTransformer(
        normalize_categories, validate=False
    )),
    ("imputer", SimpleImputer(
        strategy="constant", fill_value="MISSING"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        dtype=np.float32
    )),
])

# Training data only
X_train_cat = (
    train_prepared_lf
    .select(categorical_features)
    .collect()
    .to_pandas()
)

categorical_pipeline.fit(X_train_cat)

encoder = categorical_pipeline.named_steps["encoder"]

print("Categorical features:", categorical_features)
print("Encoded columns:", len(
    encoder.get_feature_names_out(categorical_features)
))

for name, categories in zip(
    categorical_features, encoder.categories_
):
    print(f"{name}: {len(categories)} categories")
    print(" ", categories)

# Check a tiny transformed sample
sample = categorical_pipeline.transform(X_train_cat.head(5))

print("\nSample encoded shape:", sample.shape)
print("Encoded dtype:", sample.dtype)
print("Categorical preprocessing: PASS")


Categorical features: ['home_ownership', 'addr_state']
Encoded columns: 55
home_ownership: 4 categories
  ['ANY' 'MORTGAGE' 'OWN' 'RENT']
addr_state: 51 categories
  ['AK' 'AL' 'AR' 'AZ' 'CA' 'CO' 'CT' 'DC' 'DE' 'FL' 'GA' 'HI' 'IA' 'ID'
 'IL' 'IN' 'KS' 'KY' 'LA' 'MA' 'MD' 'ME' 'MI' 'MN' 'MO' 'MS' 'MT' 'NC'
 'ND' 'NE' 'NH' 'NJ' 'NM' 'NV' 'NY' 'OH' 'OK' 'OR' 'PA' 'RI' 'SC' 'SD'
 'TN' 'TX' 'UT' 'VA' 'VT' 'WA' 'WI' 'WV' 'WY']

Sample encoded shape: (5, 55)
Encoded dtype: float32
Categorical preprocessing: PASS



## 02.11 — Combined Preprocessing Pipeline

Combine training-fitted preprocessing components:

- 51 numerical features: median imputation.
- 3 missingness indicators: preserve binary values.
- 2 categorical features: missing-value handling
  and one-hot encoding.

Fit the ColumnTransformer on training data only.
Verify output dimensions, data types and finite values.

Scaling and optional PCA will be evaluated separately.
No model training is performed here.


In [11]:

# 02.11 — Combine preprocessing components

import numpy as np
from scipy import sparse
from sklearn.compose import ColumnTransformer

indicator_features = [
    "delinq_recency_missing",
    "emp_length_missing",
    "recent_inq_missing",
]

input_features = (
    numeric_input_features
    + indicator_features
    + categorical_features
)

assert len(input_features) == len(set(input_features))

# One reusable preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_input_features),
        ("indicators", "passthrough", indicator_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

# Read only selected prepared training features
X_train = (
    train_prepared_lf
    .select(input_features)
    .collect()
    .to_pandas()
)

assert len(X_train) == 457355
assert not X_train[indicator_features].isna().any().any()

# Fit on TRAIN ONLY
preprocessor.fit(X_train)

# Verify using a small transformed sample
X_sample = preprocessor.transform(X_train.head(2000))

if sparse.issparse(X_sample):
    values = X_sample.data
else:
    values = np.asarray(X_sample)

assert np.isfinite(values).all(), (
    "Non-finite values found after preprocessing"
)

# Retrieve encoded dimensions
fitted_encoder = (
    preprocessor
    .named_transformers_["categorical"]
    .named_steps["encoder"]
)

encoded_names = fitted_encoder.get_feature_names_out(
    categorical_features
)

expected_columns = (
    len(numeric_input_features)
    + len(indicator_features)
    + len(encoded_names)
)

assert X_sample.shape[1] == expected_columns

print("Training rows:", len(X_train))
print("Original inputs:", len(input_features))
print("Numerical:", len(numeric_input_features))
print("Indicators:", len(indicator_features))
print("One-hot columns:", len(encoded_names))
print("Final model columns:", expected_columns)

print("\nSample shape:", X_sample.shape)
print("Output dtype:", X_sample.dtype)
print("Sparse output:", sparse.issparse(X_sample))
print("Finite-value check: PASS (sample)")
print("Preprocessor fitted on training data only.")


Training rows: 457355
Original inputs: 56
Numerical: 51
Indicators: 3
One-hot columns: 55
Final model columns: 109

Sample shape: (2000, 109)
Output dtype: float64
Sparse output: False
Finite-value check: PASS (sample)
Preprocessor fitted on training data only.



## 02.12 — Model-Specific Scaling

Prepare separate preprocessing configurations:

- **Linear models:** Median imputation followed by StandardScaler.
- **Tree models:** Median imputation without scaling.
- **Categorical features:** One-Hot Encoding for both.
- **Missingness indicators:** Preserve as binary values.

Fit scaling parameters exclusively on training data.

PCA remains an optional experiment after establishing
the non-PCA Logistic Regression baseline.


In [ ]:

# 02.12 — Scaling for linear models

import numpy as np
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Logistic Regression: imputation + scaling
numeric_pipeline_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline_lr, numeric_input_features),
        ("indicators", "passthrough", indicator_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

# Fit on TRAIN ONLY
preprocessor_lr.fit(X_train)

# Existing preprocessor remains unscaled for tree models
preprocessor_tree = preprocessor

# Check a small sample
sample = X_train.head(1000)

X_lr_sample = preprocessor_lr.transform(sample)
X_tree_sample = preprocessor_tree.transform(sample)

def check_matrix(X):
    values = X.data if sparse.issparse(X) else np.asarray(X)
    return np.isfinite(values).all()

assert check_matrix(X_lr_sample)
assert check_matrix(X_tree_sample)
assert X_lr_sample.shape == X_tree_sample.shape

print("Logistic Regression:", X_lr_sample.shape)
print("Decision Tree / RF:", X_tree_sample.shape)

print("LR numerical scaling:",
      preprocessor_lr.named_transformers_["numeric"]
      .named_steps["scaler"])

print("LR dtype:", X_lr_sample.dtype)
print("Tree dtype:", X_tree_sample.dtype)
print("Finite values: PASS")


Logistic Regression: (1000, 109)
Decision Tree / RF: (1000, 109)
LR numerical scaling: StandardScaler()
LR dtype: float64
Tree dtype: float64
Finite values: PASS



## 02.13 — PCA Explained-Variance Analysis

Investigate dimensionality reduction on the 51 numerical features.

- Use a reproducible 50,000-row training sample.
- Reuse training-fitted imputation and scaling.
- Examine cumulative explained variance.
- Determine components needed for 90%, 95%, and 99% variance.

Exclude categorical features and missingness indicators
from PCA.

PCA is exploratory; retain the original-feature baseline
until validation establishes whether PCA is beneficial.


In [14]:

# 02.13 — Training-only PCA investigation

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from IPython.display import display

# Reproducible training sample
X_pca_sample = X_train[numeric_input_features].sample(
    n=min(50_000, len(X_train)),
    random_state=42
)

# Reuse already fitted numerical imputer + scaler
numeric_transformer = preprocessor_lr.named_transformers_["numeric"]
X_scaled = numeric_transformer.transform(X_pca_sample)

assert np.isfinite(X_scaled).all()

# Fit PCA only on the training sample
pca_probe = PCA(svd_solver="full")
pca_probe.fit(X_scaled)

cumulative = np.cumsum(pca_probe.explained_variance_ratio_)

# Summary of components needed
pca_summary = pd.DataFrame({
    "variance_retained": ["90%", "95%", "99%"],
    "components_required": [
        int(np.searchsorted(cumulative, threshold) + 1)
        for threshold in [0.90, 0.95, 0.99]
    ]
})

print("Numerical features:", X_scaled.shape[1])
print("Training sample:", X_scaled.shape[0])

display(pca_summary)

# Variance explained by every component
pca_variance = pd.DataFrame({
    "component": np.arange(1, len(cumulative) + 1),
    "explained_pct": 100 * pca_probe.explained_variance_ratio_,
    "cumulative_pct": 100 * cumulative
})

print("\nFirst 10 components:")
display(pca_variance.head(10).round(2))

# Save for later model comparison
REPORT_DIR.mkdir(parents=True, exist_ok=True)

pca_variance.to_csv(
    REPORT_DIR / "02_pca_explained_variance.csv",
    index=False
)


Numerical features: 51
Training sample: 50000


,variance_retained,components_required
0,90%,27
1,95%,32
2,99%,41



First 10 components:


,component,explained_pct,cumulative_pct
0,1,15.61,15.61
1,2,9.19,24.80
2,3,7.24,32.04
3,4,6.39,38.43
4,5,5.31,43.74
5,6,4.28,48.02
6,7,3.89,51.91
7,8,3.58,55.49
8,9,3.01,58.50
9,10,2.83,61.33



**PCA findings (training sample: 50,000 loans):**

- 90% variance: 27/51 components.
- 95% variance: 32/51 components.
- 99% variance: 41/51 components.
- First 10 components explain 61.33% variance.

**Decision:** Moderate numerical redundancy exists.
Retain original features for baseline modeling.
Compare Logistic Regression with and without 95% PCA
using validation data.

Exclude missingness indicators and one-hot categorical
features from PCA. No model or source data modified.



## 02.14 — Financial Feature Engineering Feasibility

Investigate two candidate features using training data only:

1. **Loan-to-income ratio:** Requested loan amount divided
   by annual income. Check zero income and extreme ratios.
2. **Credit-history age:** Time between the earliest credit
   line and loan issue date. Inspect date formats and
   missingness before conversion.

Do not remove outliers or modify the original features.

Engineered features remain provisional until their
contribution is evaluated on validation data.


In [15]:

# 02.14 — Financial feature-engineering feasibility

# A. Loan-to-income ratio (training only)
ratio_lf = train_prepared_lf.with_columns(
    pl.when(pl.col("annual_inc") > 0)
    .then(pl.col("loan_amnt") / pl.col("annual_inc"))
    .otherwise(None)
    .alias("loan_to_income")
)

ratio_summary = ratio_lf.select(
    pl.len().alias("rows"),
    (pl.col("annual_inc") <= 0)
    .fill_null(False).sum().alias("nonpositive_income"),
    pl.col("loan_to_income").null_count().alias("ratio_missing"),
    pl.col("loan_to_income").median().alias("median"),
    pl.col("loan_to_income").quantile(0.95).alias("p95"),
    pl.col("loan_to_income").quantile(0.99).alias("p99"),
    (pl.col("loan_to_income") > 1)
    .fill_null(False).sum().alias("ratio_above_1"),
).collect()

print("A. LOAN-TO-INCOME RATIO")
display(ratio_summary.to_pandas())


# B. Earliest credit-line date feasibility
credit_raw = pl.col("earliest_cr_line").cast(
    pl.String
).str.strip_chars()

credit_summary = train_lf.select(
    pl.len().alias("rows"),
    (
        credit_raw.is_null() | (credit_raw == "")
    ).fill_null(True).sum().alias("missing_dates"),
    credit_raw.n_unique().alias("unique_values"),
).collect()

credit_samples = (
    train_lf
    .select(credit_raw.alias("earliest_cr_line"))
    .unique()
    .drop_nulls()
    .head(15)
    .collect()
)

print("\nB. EARLIEST CREDIT-LINE DATE")
display(credit_summary.to_pandas())

print("\nSample date values:")
display(credit_samples.to_pandas())


A. LOAN-TO-INCOME RATIO


,rows,nonpositive_income,ratio_missing,median,p95,p99,ratio_above_1
0,457355,0,0,0.183333,0.4,0.455828,0



B. EARLIEST CREDIT-LINE DATE


,rows,missing_dates,unique_values
0,457355,0,673



Sample date values:


,earliest_cr_line
0,Jan-1987
1,Jun-2008
2,Sep-1989
3,Nov-1992
4,Aug-2002
5,Jan-1999
6,Oct-1998
7,Jan-2001
8,Oct-2001
9,May-2009



## 02.15 — Credit-History Age Validation

Derive credit-history age from `earliest_cr_line`
and `issue_date`, using training data only.

Check:
- Date parsing failures.
- Credit-history dates after loan issuance.
- Unusually long credit histories.
- Distribution of credit-history age.

Preserve the original data. Keep the engineered feature
provisional until validation-based model comparison.


In [16]:

# 02.15 — Credit-history age validation (train only)

credit_text = (
    pl.col("earliest_cr_line")
    .cast(pl.String)
    .str.strip_chars()
)

credit_date = (
    pl.concat_str([credit_text, pl.lit("-01")])
    .str.strptime(pl.Date, "%b-%Y-%d", strict=False)
)

credit_age_review_lf = (
    train_lf
    .with_columns(
        credit_date.alias("_credit_start_date")
    )
    .with_columns(
        (
            (pl.col("issue_date").dt.year()
             - pl.col("_credit_start_date").dt.year()) * 12
            + pl.col("issue_date").dt.month()
            - pl.col("_credit_start_date").dt.month()
        ).alias("credit_history_months")
    )
    .with_columns(
        (
            pl.col("credit_history_months") / 12
        ).cast(pl.Float32).alias("credit_history_years")
    )
)

credit_age_summary = credit_age_review_lf.select(
    pl.len().alias("rows"),

    pl.col("_credit_start_date")
    .null_count().alias("unparsed_dates"),

    (pl.col("credit_history_months") < 0)
    .fill_null(False).sum().alias("future_credit_dates"),

    (pl.col("credit_history_years") > 50)
    .fill_null(False).sum().alias("history_above_50_years"),

    pl.col("credit_history_years")
    .min().alias("min_years"),

    pl.col("credit_history_years")
    .median().alias("median_years"),

    pl.col("credit_history_years")
    .quantile(0.95).alias("p95_years"),

    pl.col("credit_history_years")
    .max().alias("max_years"),
).collect()

print("CREDIT-HISTORY AGE VALIDATION")
display(credit_age_summary.to_pandas())

# Show invalid records only if found
invalid_credit_dates = credit_age_review_lf.filter(
    pl.col("_credit_start_date").is_null()
    | (pl.col("credit_history_months") < 0)
)

display(
    invalid_credit_dates.select(
        "earliest_cr_line",
        "issue_date",
        "_credit_start_date",
        "credit_history_months",
    ).limit(10).collect().to_pandas()
)


CREDIT-HISTORY AGE VALIDATION


,rows,unparsed_dates,future_credit_dates,history_above_50_years,min_years,median_years,p95_years,max_years
0,457355,0,0,387,3.0,14.75,30.916666,71.0


,earliest_cr_line,issue_date,_credit_start_date,credit_history_months



**Findings:**
- All 457,355 credit-history dates parsed successfully.
- No credit-history date occurred after loan issuance.
- Median credit-history age: 14.75 years.
- 387 loans (0.085%) have credit histories above 50 years.
- Maximum observed credit-history age: 71 years.

**Decision:** Retain `credit_history_years` as a
provisional engineered feature. No outlier removal or
capping is justified.

Together with `loan_to_income`, evaluate its incremental
predictive value using validation data only.



## 02.16 — Integrate Financial Feature Engineering

Extend the existing feature preparation function with two
optional engineered features:

- `loan_to_income`: Loan amount divided by annual income.
- `credit_history_years`: Credit-history age relative to issue date.

Maintain two configurations:
- Baseline: 109 transformed columns.
- Enhanced: 111 transformed columns.

Use consistent deterministic transformations across all
splits. Both engineered features remain provisional until
validation demonstrates their usefulness.


In [19]:

# 02.16 — Baseline vs enhanced feature configurations

engineered_features = [
    "loan_to_income",
    "credit_history_years",
]

# Baseline views remain unchanged
train_prepared_lf = prepare_features(train_lf, include_engineered=False)
val_prepared_lf = prepare_features(val_lf, include_engineered=False)
test_prepared_lf = prepare_features(test_lf, include_engineered=False)

# Enhanced views contain two additional numerical features
train_enhanced_lf = prepare_features(train_lf, include_engineered=True)
val_enhanced_lf = prepare_features(val_lf, include_engineered=True)
test_enhanced_lf = prepare_features(test_lf, include_engineered=True)

# Preserve the existing baseline feature list
numeric_input_features_enhanced = (
    numeric_input_features + engineered_features
)

baseline_columns = set(
    numeric_input_features
    + indicator_features
    + categorical_features
    + ["target", "issue_date"]
)

enhanced_columns = baseline_columns | set(engineered_features)

assert set(train_prepared_lf.collect_schema().names()) == baseline_columns
assert set(train_enhanced_lf.collect_schema().names()) == enhanced_columns

# Check engineered features on training data
engineering_summary = train_enhanced_lf.select(
    pl.len().alias("train_rows"),
    pl.col("loan_to_income").null_count().alias("ratio_missing"),
    pl.col("credit_history_years").null_count().alias("history_missing"),
    pl.col("loan_to_income").median().alias("median_ratio"),
    pl.col("credit_history_years").median().alias("median_history_years"),
).collect()

display(engineering_summary.to_pandas())

print("Baseline numerical features:", len(numeric_input_features))
print("Enhanced numerical features:", len(numeric_input_features_enhanced))
print("Engineered features:", engineered_features)
print("Baseline and enhanced views: READY")


,train_rows,ratio_missing,history_missing,median_ratio,median_history_years
0,457355,0,0,0.183333,14.75


Baseline numerical features: 51
Enhanced numerical features: 53
Engineered features: ['loan_to_income', 'credit_history_years']
Baseline and enhanced views: READY



## 02.17 — Enhanced Preprocessing Configuration

Create enhanced preprocessing pipelines with two
additional financial features.

- Logistic Regression: Median imputation + StandardScaler.
- Decision Tree / Random Forest: Median imputation only.
- Categorical variables: One-Hot Encoding.
- Missingness indicators: Pass through unchanged.

Fit preprocessing on training data only.

Preserve the original 109-column baseline for later
comparison against the 111-column enhanced version.


In [20]:

# 02.17 — Enhanced preprocessing pipelines

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from scipy import sparse
import numpy as np

enhanced_input_features = (
    numeric_input_features_enhanced
    + indicator_features
    + categorical_features
)

assert len(enhanced_input_features) == 58
assert len(set(enhanced_input_features)) == 58

# Collect enhanced training inputs
X_train_enhanced = (
    train_enhanced_lf
    .select(enhanced_input_features)
    .collect()
    .to_pandas()
)

# Logistic Regression enhanced preprocessor
preprocessor_lr_enhanced = ColumnTransformer(
    transformers=[
        ("numeric", clone(numeric_pipeline_lr),
         numeric_input_features_enhanced),

        ("indicators", "passthrough", indicator_features),

        ("categorical", clone(categorical_pipeline),
         categorical_features),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

# Tree / Random Forest enhanced preprocessor
preprocessor_tree_enhanced = ColumnTransformer(
    transformers=[
        ("numeric", clone(numeric_pipeline),
         numeric_input_features_enhanced),

        ("indicators", "passthrough", indicator_features),

        ("categorical", clone(categorical_pipeline),
         categorical_features),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

# Fit only on training observations
preprocessor_lr_enhanced.fit(X_train_enhanced)
preprocessor_tree_enhanced.fit(X_train_enhanced)

# Verify small sample
sample = X_train_enhanced.head(1000)

X_lr_enhanced_sample = preprocessor_lr_enhanced.transform(sample)
X_tree_enhanced_sample = preprocessor_tree_enhanced.transform(sample)

def verify_finite(X):
    values = X.data if sparse.issparse(X) else np.asarray(X)
    return np.isfinite(values).all()

assert verify_finite(X_lr_enhanced_sample)
assert verify_finite(X_tree_enhanced_sample)

assert X_lr_enhanced_sample.shape == (1000, 111)
assert X_tree_enhanced_sample.shape == (1000, 111)

print("Enhanced training rows:", len(X_train_enhanced))
print("Original baseline model columns: 109")

print("\nEnhanced configurations:")
print("Logistic Regression:", X_lr_enhanced_sample.shape)
print("Decision Tree / RF:", X_tree_enhanced_sample.shape)

print("\nEnhanced feature engineering:",
      engineered_features)

print("\nFinite-value validation: PASS")
print("Original baseline preprocessors preserved.")


Enhanced training rows: 457355
Original baseline model columns: 109

Enhanced configurations:
Logistic Regression: (1000, 111)
Decision Tree / RF: (1000, 111)

Enhanced feature engineering: ['loan_to_income', 'credit_history_years']

Finite-value validation: PASS
Original baseline preprocessors preserved.
